# 最適執行ビジュアルラボ

## エグゼクティブサマリー
執行を速めるほど市場インパクトは増え、遅くするほど価格変動（タイミング）リスクが増えます。本ノートブックは、この基本的なトレードオフを、単一の再現可能な合成市場のうえで一貫して扱います。Almgren–Chriss のスケジューリング、板の回復力（resilient liquidity）、反応型指値注文板、取引コスト分析（TCA）、制約付き強化学習による調整を、共通乱数で対にして比較します。強化学習の整形報酬と経済的な実装ショートフォールは、常に明確に分離します。

*本ページは日本語版です（英語版: `01_optimal_execution_visual_lab.html`）。*

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from optimal_execution.config import load_config
from optimal_execution.provenance import artifact_dirs

config_path = Path(os.environ.get("OPTIMAL_EXECUTION_CONFIG", "configs/quick.yaml"))
cfg = load_config(config_path)
paths = artifact_dirs(cfg)
display(
    Markdown(
        f"**プロファイル:** `{cfg.profile}` · **シード:** `{cfg.seed}` · **注文:** `{cfg.initial_inventory:,.0f}` 株 · **時間軸:** `{cfg.horizon_seconds:.0f}` 秒"
    )
)

## コンセプトマップと市場マイクロストラクチャーの定義

**スプレッド**は最良気配をまたいで即時に約定する費用、**流動性**は表示深度と補充力の総合、**マーケットインパクト**は自分の取引が価格に残す足跡、**タイミングリスク**は不動（unaffected）価格が動くなかで在庫を持ち続けることの不確実性です。**執行方策**は、時刻・在庫・板の状態を、上限付きの成行・指値注文へ写像します。

戦略的ベースライン（AC ／ 回復力スケジュール）→ 戦術的ヒューリスティックまたは RL 調整 → 安全層（在庫・参加率・価格カラー・期限）。

In [ ]:
display(
    pd.DataFrame(
        {
            "variable": [
                "side",
                "arrival price",
                "initial inventory",
                "steps",
                "annualized volatility",
                "ADV",
                "spread",
                "participation cap",
            ],
            "value": [
                cfg.side,
                cfg.arrival_price,
                cfg.initial_inventory,
                cfg.n_decision_steps,
                cfg.annualized_volatility,
                cfg.average_daily_volume,
                cfg.spread_bps,
                cfg.max_participation_rate,
            ],
            "unit": [
                "",
                "price/share",
                "shares",
                "",
                "fraction/year",
                "shares/day",
                "bps",
                "fraction",
            ],
        }
    )
)

## 合成市場プロファイルと不動価格パス
出来高・ボラティリティ・スプレッド・深度は、局所的な確率乗数を伴う日中プロファイルに従います。不動（unaffected）価格には自己インパクトが一切含まれず、被影響価格はその上に重ねられます。共通乱数により、戦略間の差は独立ではなく対（ペア）として測られます。

In [ ]:
display(Image(filename=str(paths["figures"] / "unaffected_price_paths.png")))

## 一時的・恒久的・過渡的インパクトと平方根則

3 つの線形経路を区別します。**一時的（temporary）**インパクトは取引中だけ支払う価格譲歩で、1 株あたり `η · v`（`v = q/dt` は株数/秒）、1 ステップの現金コストは `η q² / dt`。取引が終われば痕跡を残しません。**恒久的（permanent）**インパクトは仲値の恒久シフト `γ · (累積取引株数)` で、減衰せず、後続の取引が引き継ぐ足跡です。**過渡的（transient）**インパクトは、取引で積み上がり回復率 ρ で緩和する変位状態 `D` です。離散の点インパルス則は `D_{k+1} = exp(−ρ dt) (D_k + η_t q_k)`、ステップ k の約定は取引前の `D_k` に自分のジャンプの半分 `0.5 η_t q_k`（板を通した区間平均）を加えた分を負担します。符号規約：各量は正の不利量で、売りは `S0 − 不利量`、買いは `S0 + 不利量` で約定します。

指数則は伝播核（propagator）`D_k = Σ_{j<k} G((k−j) dt) η_t q_j`（`G(0) = 1`、指数核または完全単調な冪核）の ρ 特殊ケースです。シミュレータがこれらの潜在状態を公開しているため、各取引の一時的／恒久的／過渡的への分解はモデル内で厳密になります——実データでは決して得られない特権です。

平方根則 `I(Q) = Y σ_day √(Q / ADV)` は、サイズ Q のメタオーダーの**総**インパクトに対する凹関数で、経験的診断としてのみ示し、線形経路の上には加算しません（二重計上を避けるため）。

In [ ]:
for name in ["impact_channels.png", "impact_recovery.png", "sqrt_impact.png"]:
    display(Image(filename=str(paths["figures"] / name)))

## Almgren–Chriss の導出・感応度・効率的フロンティア

区間 [0, T] で X 株を売る、連続時間の平均–分散プログラムです。線形の一時的インパクト η と価格ボラティリティ σ のもとで、在庫パス x(t)（x(0) = X, x(T) = 0）について `J[x] = ∫_0^T [η ẋ(t)² + λ σ² x(t)²] dt` を最小化します。第 1 項は速く売ること（ẋ は取引速度）のインパクトコスト、第 2 項は在庫 x を保持したまま価格が拡散するタイミングリスクで、リスク回避度 λ で重み付けされます。

Euler–Lagrange 条件は `ẍ = κ² x`（緊急度 `κ = √(λ σ² / η)`）を与え、境界条件を満たす解は `x*(t) = X · sinh(κ(T − t)) / sinh(κ T)` です。`κ → 0`（リスク中立、またはリスクが安い極限）では直線 `X(1 − t/T)`、すなわち TWAP に退化します。λ や σ が大きい（緊急度が高い）ほど執行は前倒しに、η が大きい（インパクトが高価）ほど TWAP 方向へ平坦化します。注文サイズ X は数量をスケールしますが、正規化した形は変えません——線形設定の帰結です。

決定グリッド上の子注文は `q_k = x_k − x_{k+1}`。期待ショートフォールは `E[C] = (γ/2) X² + η Σ q_k² / dt`（＋ ハーフスプレッド · X ＋ 手数料）、タイミングリスク分散は `V[C] = Σ σ_k² dt · x_{k+1}²`（各ステップの価格変動に晒される在庫）です。λ を掃引すると、期待コストとタイミングリスク標準偏差の**効率的フロンティア**——最適な速度／リスクのトレードオフの一覧——が描けます。

In [ ]:
display(Image(filename=str(paths["figures"] / "ac_trajectories.png")))
display(Image(filename=str(paths["figures"] / "efficient_frontier.png")))

## Obizhaeva–Wang 型の回復力のある流動性

AC が速度を一時的インパクト率で罰するのに対し、OW は**回復力のある**板をモデル化します。取引は純粋に過渡的な変位 `D` を押し上げ、率 ρ で減衰します（`dD = −ρ D dt + η_t v dt`、離散の点インパルスは `D_{k+1} = e^{−ρ dt}(D_k + η_t q_k)`）。ブロック形状の板に対して執行すると、スケジュールの期待コストは二次形式 `C(q) = η_t [½ Σ q_k² + Σ_{j<k} q_k q_j G((k−j) dt)] = ½ η_t q' M q` になります。核行列は `M_kj = G(|k−j| dt)`、`M_kk = 1`。指数核（および完全単調な冪核）では M は対称正定値なので、スケジューリング QP は良設定です。

2 つの解法が一致します。古典的なリスク中立 Obizhaeva–Wang（2013）閉形式は、t = 0 と t = T に等しいブロック `X / (ρT + 2)` を置き、間を一定率 `ρ X / (ρT + 2)` で取引します。離散ソルバは `Σ q_k = X, q ≥ 0` のもとで `½ η_t q'Mq` を最小化し、`q = M⁻¹ 1 · X / (1' M⁻¹ 1)` を小さな active-set ループで解きます（任意の核で有効）。回復力が低い（ρ が小さい）と変位が持続し忍耐が有利に、ρ が高いと流動性が補充され再利用が速くできます。

本ラボはリスク中立の連続解と離散凸最適化のみを実装します。OW の**リスク回避**拡張は実装していません——リスク選好は本スイートでは Almgren–Chriss 層を通じてのみ入ります（METHODOLOGY.md 参照）。

In [ ]:
resilience = pd.read_csv(paths["data"] / "resilience_sweep.csv")
display(resilience.groupby(["rho", "method"], as_index=False)[["cost", "twap_cost"]].first())

## 反応型指値注文板・注文不均衡・イベント状態

簡略化した板は、片側あたり 1 つの集約 Level-1 気配に深い流動性密度を加えた構造で、スプレッド・注文不均衡・直近フロー・ボラティリティ・自己インパクトを追跡します。**不動**仲値 `s0` は自分の取引に反応しません。**被影響**フェア仲値は `mid = s0 − sign · (γ · 累積成行 + D)`、すなわち古典世界と同じ恒久 γ と過渡 D の経路です（受動的な指値約定は流動性を供給するので寄与しません）。ここでの一時的インパクトは内生的で、大口の成行注文が L1 深度を消費し、深い流動性まで板を歩き、スプレッドを広げます——別途の η は支払いません。

サブステップあたりの外生成行出来高は `count(Poisson) · 平均サイズ` で、強度は不均衡・直近リターン・ストレス・潜在的な短期アルファの cap 付き log-linear 関数です。この潜在アルファが注文フローと次の仲値変動の両方を駆動し、これが定型化された逆選択（adverse selection）の経路になります。事前抽選した一様乱数が戦略間の共通乱数を与えるため差は対で測られ、状態依存は強度だけに入ります。

対照（リプレイ、`reactive=False`）は同じ外生抽選を再利用しつつ、自己の痕跡を完全に取り除きます——深度は消費されず、気配と仲値は動かず、インパクト状態も不変です。反応型とリプレイの差が、自己の足跡だけを切り出します。これは透明な反応型サンドボックスであり、ヒストリカル・リプレイでも取引所級のリアリズムでもありません。

In [ ]:
display(Image(filename=str(paths["figures"] / "lob_depth.png")))
display(Image(filename=str(paths["figures"] / "queue_imbalance.png")))

## 成行の板歩き・指値キュー・約定確率・逆選択
成行注文は即時性を買いますが、スプレッドと板歩きのコストを払います。受動的な指値はスプレッドを取り込めますが、前に並ぶキューが約定確率を下げ、期限時の後始末（cleanup）が支配的になることもあります。潜在的な短期アルファが注文フローと次の価格変動を同時に傾け、定型化された逆選択を生みます。

In [ ]:
display(Image(filename=str(paths["figures"] / "market_vs_limit.png")))
lob_summary = pd.read_csv(paths["metrics"] / "lob_strategy_summary.csv")
display(
    lob_summary[["strategy_id", "is_mean_bps", "cvar95_bps", "fill_rate", "cleanup_qty"]].round(3)
)

## 静的スケジュール・実装ショートフォール分布・TCA 分解・テールリスク
即時・TWAP・VWAP 型・POV・Almgren–Chriss・回復力考慮の各スケジュールは、同一の確率パスを共有します。実装ショートフォールが正なら、売り・買いのどちらでも不利です。潜在的なタイミング・スプレッド・一時的・恒久的・過渡的・手数料・後始末の各成分は、この合成モデル内でのみ厳密です。

In [ ]:
for name in ["strategy_schedules.png", "is_distributions.png", "tca_decomposition.png"]:
    display(Image(filename=str(paths["figures"] / name)))
classical = pd.read_csv(paths["metrics"] / "classical_strategy_summary.csv")
display(
    classical[
        ["strategy_id", "is_mean_bps", "is_mean_ci_lo_bps", "is_mean_ci_hi_bps", "cvar95_bps"]
    ].round(3)
)

## 反応型と非反応型のシミュレーション
対照ランは同じ外生抽選を再利用しつつ、自己の足跡を取り除きます。生じるコスト差は、設定した注文サイズについてこの簡易リプレイが過小評価する分を定量化します——実市場の推定値ではありません。

In [ ]:
display(Image(filename=str(paths["figures"] / "reactive_vs_replay.png")))
display(
    pd.read_csv(paths["metrics"] / "reactive_comparison.csv")[
        ["mode", "is_mean_bps", "cvar95_bps"]
    ].round(3)
)

## RL の状態・行動・報酬・安全層・学習曲線

1 エピソードは 1 つの親注文で、エージェントは `n_decision_steps` 回の意思決定を行い、その合間に板が進行します。12 個の正規化観測は、時刻・在庫・スプレッド・両側深度・不均衡・直近リターンとフロー・ボラティリティ・過渡的インパクト・出来高・未約定指値数量です。離散行動空間は 15 = 5 × 3：上限付きの成行倍率 `m ∈ {0, 0.5, 1, 1.5, 2}` × ベースライン枚数 を、指値指令 `l ∈ {none, join, improve}` と掛け合わせます。

**残差 RL** は Almgren–Chriss スケジュールをベースラインに使うため、`m = 1` は古典方策を厳密に再現し、ネットワークはその周りの上限付き逸脱を学びます。**自由 RL** は TWAP 相当のベースラインを使います。小さな共有ボディの actor–critic MLP を PPO（クリップ代理目的 ＋ GAE）で、**整形**報酬（増分コスト＋在庫・インパクト罰則）に対して学習します。重要な点として、検証と報告するすべての比較は、互いに素なホールドアウトのシードでの**経済的**実装ショートフォールを使います——整形報酬を損益として提示することはありません。

硬い安全層は、スクリプト方策でも学習方策でも一貫して支配的です：過剰執行を禁止し、子注文サイズと参加率を実現出来高の EMA に対して上限付けし、成行に価格カラーを適用し、期限で強制清算します（上限は免除だが板を歩くため懲罰的）。違反は計数・罰則化されるため、制約を「ごまかす」ことは決して報われません。

In [ ]:
display(Image(filename=str(paths["figures"] / "rl_training_history.png")))
history = pd.read_csv(paths["metrics"] / "rl_training_history.csv")
display(
    history.groupby("run_id", as_index=False)
    .tail(1)[["run_id", "episodes", "train_is_ma_bps", "best_val_is_bps"]]
    .round(3)
)

## 分布内比較とストレステスト
TWAP・Almgren–Chriss・混合ヒューリスティック・残差 RL・自由 RL を、固定したホールドアウトのシードで評価します。ストレス・レジームはボラティリティ・流動性・出来高プロファイル・不利アルファ・スプレッド／回復力を変化させます。quick プロファイルは RL シードが 1 つだけなので、シードに頑健な優位性は確立できません。

In [ ]:
display(Image(filename=str(paths["figures"] / "stress_test_heatmap.png")))
stress = pd.read_csv(paths["metrics"] / "stress_summary.csv")
display(stress.pivot(index="strategy_id", columns="regime", values="is_mean_bps").round(3))

## 特徴量アブレーション・RL 行動診断・分布シフト
各残差方策は、観測グループを 1 つずつマスクしたうえで再学習します。モデル誤指定テストは、学習済み方策を固定したまま回復力・深度・スプレッドストレス・流動性を変えます。差はシミュレータ上の証拠であって、実市場での因果的な特徴量重要度ではありません。

In [ ]:
display(Image(filename=str(paths["figures"] / "feature_ablation.png")))
display(Image(filename=str(paths["figures"] / "model_misspecification.png")))
display(
    pd.read_csv(paths["metrics"] / "ablation_summary.csv")[
        ["strategy_id", "feature_removed", "is_mean_bps", "delta_vs_full_bps"]
    ].round(3)
)

## この実験が示すこと
記録済みシードのもとで実装が再現可能であること、古典的な緊急度がリスクとインパクトのパラメータに正しく反応すること、エージェントが自分の合成市場を変えること、指値注文がスプレッドと約定リスクを交換すること、方策の順位がストレスとモデルシフトのもとで変わり得ること——これらを示します。

In [ ]:
best_classical = classical.loc[classical.is_mean_bps.idxmin()]
best_lob = lob_summary.loc[lob_summary.is_mean_bps.idxmin()]
id_test = stress[stress.regime == "in_distribution"]
best_id = id_test.loc[id_test.is_mean_bps.idxmin()]
display(
    Markdown(f"""**Generated quick-run findings**  
- Lowest classical mean cost: `{best_classical.strategy_id}` at **{best_classical.is_mean_bps:.3f} bps**.  
- Lowest reactive-LOB mean cost: `{best_lob.strategy_id}` at **{best_lob.is_mean_bps:.3f} bps**.  
- Lowest in-distribution comparison cost: `{best_id.strategy_id}` at **{best_id.is_mean_bps:.3f} bps**.  
These rankings are conditional on the synthetic calibration.""")
)

## この実験が示さないこと・限界と次の一歩
本ラボは、実市場での収益性・因果的な特徴量価値・取引所級のリアリズム・統計的な RL 優位性を示しません。隠れ流動性・レイテンシ・複数会場ルーティング・戦略的な対抗者・相場操縦制約・クロスインパクト・経験的キャリブレーションを省いています。次の一歩は、厳格なホールドアウトでの実データ較正、マルチアセットのクロスインパクト、ディーラー／RFQ 市場、確率的な回復力、より安全な制約付き RL、分布的にロバストな評価、そしてラフボラティリティや Hawkes フローとの相互作用です。